# 12.4 - Itertools => Chaining, Zipping & Mapping

## `chain(*iterables)`

Joins several iterables into one **lazy** stream.

```python
from itertools import chain

list(chain("ABC", "DEF"))            # ['A', 'B', 'C', 'D', 'E', 'F']
list(chain([1, 2], (3, 4), {5}))     # works across different types
```

Unlike `+`, `chain` does **not copy** the data. It also works with any iterable, not only lists.

## `chain.from_iterable(iterable)`

Like `chain`, but takes **one** iterable that yields iterables. This is the standard way to **flatten one level**.

```python
list(chain.from_iterable([[1, 2], [3], [4, 5]]))   # [1, 2, 3, 4, 5]
```

The outer iterable is evaluated lazily, so it may even be infinite.

## `zip_longest(*iterables, fillvalue=None)`

Zips until the **longest** iterable ends. Missing values are replaced by `fillvalue`.

```python
from itertools import zip_longest

list(zip_longest("ABCD", "xy", fillvalue="-"))
# [('A', 'x'), ('B', 'y'), ('C', '-'), ('D', '-')]
```

If one input can be infinite, limit the result with `islice` or `takewhile`.

### `zip`, `zip(strict=True)` and `zip_longest`

| Tool | If lengths differ |
|---|---|
| `zip(a, b)` | Stops at the **shortest**, silently dropping the rest |
| `zip(a, b, strict=True)` | Raises `ValueError` |
| `zip_longest(a, b)` | Continues to the **longest**, filling gaps |

## `starmap(function, iterable)`

Calls `function(*args)` for every item. Use it instead of `map()` when the arguments are already grouped in tuples.

```python
from itertools import starmap

list(starmap(pow, [(2, 5), (3, 2), (10, 3)]))   # [32, 9, 1000]
```

| `map` | `starmap` |
|---|---|
| `map(f, a, b)` calls `f(a_i, b_i)` | `starmap(f, pairs)` calls `f(*pair)` |
| Arguments come from separate iterables | Arguments come pre-zipped in tuples |

## Key Rules

- `chain` and `chain.from_iterable` are lazy and never copy the data.
- Use `chain.from_iterable` to flatten **one** level. It does not flatten deeper levels.
- Prefer `zip(strict=True)` when different lengths would be a bug.
- `starmap(f, zip(a, b))` is the same as `map(f, a, b)`.

## Source

https://docs.python.org/3/library/itertools.html#itertools.chain

https://docs.python.org/3/library/itertools.html#itertools.zip_longest

https://docs.python.org/3/library/itertools.html#itertools.starmap

In [ ]:
from itertools import chain, zip_longest, starmap, count, islice
import operator

# chain
print(list(chain("ABC", "DEF")))                 # ['A', 'B', 'C', 'D', 'E', 'F']
print(list(chain([1, 2], (3, 4), {5})))          # mixed iterable types

# chain is lazy and does not copy
combined = chain(range(3), range(10, 13))
print(next(combined), next(combined))            # 0 1

# chain.from_iterable: flatten ONE level
print(list(chain.from_iterable([[1, 2], [3], [4, 5]])))     # [1, 2, 3, 4, 5]
print(list(chain.from_iterable(["ab", "cd"])))              # ['a', 'b', 'c', 'd']

# It does not flatten deeper levels
print(list(chain.from_iterable([[1, [2, 3]], [4]])))        # [1, [2, 3], 4]

# The outer iterable can be lazy (and even infinite)
lazy_outer = chain.from_iterable([n, n] for n in count(1))
print(list(islice(lazy_outer, 6)))               # [1, 1, 2, 2, 3, 3]

# zip vs zip(strict=True) vs zip_longest
a, b = "ABCD", "xy"
print(list(zip(a, b)))                           # stops at the shortest
try:
    list(zip(a, b, strict=True))
except ValueError as error:
    print("strict:", type(error).__name__)
print(list(zip_longest(a, b, fillvalue="-")))    # continues to the longest
print(list(zip_longest([1, 2, 3], [10])))        # default fillvalue is None

# starmap: call function(*args) for each item
print(list(starmap(pow, [(2, 5), (3, 2), (10, 3)])))        # [32, 9, 1000]
print(list(starmap(operator.mul, [(2, 3), (4, 5)])))        # [6, 20]

# map vs starmap
xs, ys = [1, 2, 3], [10, 20, 30]
print(list(map(operator.add, xs, ys)))                      # [11, 22, 33]
print(list(starmap(operator.add, zip(xs, ys))))             # [11, 22, 33]